# Step 3 — feature sanity-check

This notebook is the pre-Step-4 sanity gate on Harry's feature pack. It builds
the **canonical Step 3 feature matrix ad-hoc** (Step 4a will formalise the same
construction as a pipeline function), then runs a battery of diagnostics on
three representative instruments and four cross-cutting panels.

**Canonical column set.** The pack ships 22 public feature *functions*; the
wavelet function returns 5 columns, so the per-row matrix is **19 scalar
columns + 5 wavelet columns = 24 columns** *including* `time_since_last_flip`.
That feature is a deterministic transform of `signal_run_length`
(`= run_length − 1`), so the canonical model-facing set **excludes it, leaving
23 columns**. (The feature doc rounds this to "24/25" in places; the exact
shipped, non-redundant count is 23.)

**Released window.** The primary signal is released for `2020-01-03 →
2022-06-30`. Price-derived features are warmed up on OHLCV history back to
`2018-12-01` (a buffer) and then sliced to the released window; signal-derived
features are computed on the signal's native released index so their warmup is
honest. `2022-01-01` is the train/test boundary used throughout.

Sections:

1. Load data, build the 23-column matrix (per instrument).
2. Per-instrument panels for `cl1s` (energy, audit outlier), `es1s` (equity),
   `gc1s` (metals): (a) time-series with signal shading, (b) summary stats,
   (c) Spearman heatmap, (d) scale boxplot.
5. Train/test distribution shift (KS) — cross-instrument.
6. Label correlation (point-biserial) — cross-instrument.
7. Cross-instrument scale check for 5 representative features.
8. Feature-vs-signal contingency for 3 high-prior features.

A final **Findings** cell collects everything that looks off.


In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# stml is normally installed editable (`uv pip install -e .`); fall back to the
# src/ path so the notebook runs from a kernel where the package is not on
# sys.path.
try:
    import stml  # noqa: F401
except ModuleNotFoundError:
    sys.path.insert(0, str((Path.cwd() / ".." / ".." / "src").resolve()))

from stml.io import load_clean_data, load_returns_panel
from stml.harry.labels import ewma_daily_vol

from stml.harry.features.signal_trajectory import (
    signal_run_length, signal_entropy_20d, signal_flip_rate_60d, signal_cum_pnl_20d,
)
from stml.harry.features.conditional_risk import (
    expected_hit_time, prob_timeout, path_tortuosity_20d, realized_semi_vol_ratio,
)
from stml.harry.features.information_theoretic import (
    rolling_mutual_information_252d, transfer_entropy_vol_to_signal_acc,
)
from stml.harry.features.microstructure_fixed import (
    amihud_illiquidity, rolls_effective_spread, kyles_lambda, overnight_gap,
)
from stml.harry.features.cross_asset import (
    distance_to_lead_lag_centroid, asset_class_dispersion_z, ewma_implied_corr_z,
)
from stml.harry.features.wavelet import mra_energy_bands
from stml.harry.features.concept_drift import regime_alignment_score

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 160)
print(f"numpy {np.__version__} | pandas {pd.__version__}")


## 1. Load data & build the 23-column feature matrix

In [ ]:
# Resolve the repo root from the notebook location so paths work from a fresh
# kernel regardless of the launch directory.
REPO = Path.cwd()
while not ((REPO / "data").is_dir() and (REPO / "pyproject.toml").is_file()):
    if REPO.parent == REPO:
        raise FileNotFoundError("could not locate stml repo root")
    REPO = REPO.parent
DATA_DIR = REPO / "data"
EVENTS_CSV = REPO / "results" / "harry" / "events.csv"

# Window constants.
START = pd.Timestamp("2018-12-01")     # warmup buffer for price features
REL_START = pd.Timestamp("2020-01-03") # first released signal date
END = pd.Timestamp("2022-06-30")       # last released signal date
TRAIN_END = pd.Timestamp("2022-01-01") # train/test boundary

INSTRUMENTS = [
    "es1s", "nq1s", "fesx1s",
    "cl1s", "ho1s", "rb1s", "ng1s",
    "gc1s", "si1s", "hg1s", "pl1s",
]
FOCUS = ["cl1s", "es1s", "gc1s"]  # energy / equity / metals representatives

# Canonical 23-column order, grouped by family.
FEATURE_ORDER = [
    "signal_run_length", "signal_entropy_20d", "signal_flip_rate_60d",
    "signal_cum_pnl_20d",
    "rolling_mutual_information_252d", "transfer_entropy_vol_to_acc_126d",
    "expected_hit_time", "prob_timeout", "path_tortuosity_20d",
    "realized_semi_vol_ratio_20d",
    "amihud_illiquidity_20d", "rolls_spread_20d", "kyles_lambda_20d",
    "overnight_gap",
    "dist_lead_lag_centroid", "asset_class_dispersion_z", "ewma_implied_corr_z",
    "mra_energy_D1", "mra_energy_D2", "mra_energy_D3", "mra_energy_D4",
    "mra_energy_D5",
    "regime_alignment_score",
]
FEATURE_FAMILY = {
    **{c: "signal_trajectory" for c in
       ["signal_run_length", "signal_entropy_20d", "signal_flip_rate_60d",
        "signal_cum_pnl_20d"]},
    **{c: "information_theoretic" for c in
       ["rolling_mutual_information_252d", "transfer_entropy_vol_to_acc_126d"]},
    **{c: "conditional_risk" for c in
       ["expected_hit_time", "prob_timeout", "path_tortuosity_20d",
        "realized_semi_vol_ratio_20d"]},
    **{c: "microstructure" for c in
       ["amihud_illiquidity_20d", "rolls_spread_20d", "kyles_lambda_20d",
        "overnight_gap"]},
    **{c: "cross_asset" for c in
       ["dist_lead_lag_centroid", "asset_class_dispersion_z",
        "ewma_implied_corr_z"]},
    **{c: "wavelet" for c in
       ["mra_energy_D1", "mra_energy_D2", "mra_energy_D3", "mra_energy_D4",
        "mra_energy_D5"]},
    "regime_alignment_score": "concept_drift",
}
# Documented warmup *within the released window*. Price features warm up in the
# pre-2020 buffer so they are 0 here; signal features warm up inside the window.
# regime_alignment_score is special: it scores only the post-(train_end+window)
# test era by design, so it is exempted from the >5%-NaN flag.
DOC_WARMUP = {
    "signal_run_length": 0, "signal_entropy_20d": 19, "signal_flip_rate_60d": 60,
    "signal_cum_pnl_20d": 19,
    "rolling_mutual_information_252d": 251, "transfer_entropy_vol_to_acc_126d": 126,
    "expected_hit_time": 0, "prob_timeout": 0, "path_tortuosity_20d": 0,
    "realized_semi_vol_ratio_20d": 0,
    "amihud_illiquidity_20d": 0, "rolls_spread_20d": 0, "kyles_lambda_20d": 0,
    "overnight_gap": 0,
    "dist_lead_lag_centroid": 0, "asset_class_dispersion_z": 0,
    "ewma_implied_corr_z": 0,
    "mra_energy_D1": 0, "mra_energy_D2": 0, "mra_energy_D3": 0,
    "mra_energy_D4": 0, "mra_energy_D5": 0,
    "regime_alignment_score": None,
}
assert len(FEATURE_ORDER) == 23
print(f"{len(FEATURE_ORDER)} canonical feature columns across "
      f"{len(set(FEATURE_FAMILY.values()))} families")


In [ ]:
ohlcv, signals = load_clean_data(data_dir=DATA_DIR)
returns_panel = load_returns_panel(data_dir=DATA_DIR)  # date x instrument, log
sig_idx = signals.set_index("date")
print("ohlcv", ohlcv.shape, "| signals", signals.shape,
      "| returns_panel", returns_panel.shape)
print("released window:", REL_START.date(), "->", END.date())


In [ ]:
def build_feature_matrix(inst: str) -> pd.DataFrame:
    """Build the 23-column canonical feature matrix for one instrument over the
    released window.

    Signal-derived features are computed on the signal's native released index
    (so their warmup is honest); price-derived features are warmed up on OHLCV
    history from ``START`` and then sliced to the released window. Cross-asset
    features read the panel reindexed onto this instrument's trading calendar.
    ``regime_alignment_score`` is computed last, on the assembled matrix, so it
    is not self-referential.
    """
    sub = ohlcv[(ohlcv["instrument"] == inst)
                & (ohlcv["date"] >= START) & (ohlcv["date"] <= END)].sort_values("date")
    close = sub.set_index("date")["close"].astype(float)
    open_ = sub.set_index("date")["open"].astype(float)
    volume = sub.set_index("date")["volume"].astype(float)
    r = np.log(close).diff()
    sigma = ewma_daily_vol(close, span=100)        # causal EWMA daily vol
    r_trail10 = r.rolling(10).sum()                 # causal 10d return, the
                                                    # observable-at-t "forward" proxy
    rp = returns_panel.reindex(close.index)         # panel on inst's calendar

    rel_idx = close.loc[REL_START:END].index        # instrument trading days, released
    s = sig_idx[inst].astype(float).reindex(close.index).loc[REL_START:END]
    assert s.isna().sum() == 0, f"{inst}: signal has gaps on its trading calendar"
    r_s = r.reindex(rel_idx)
    sigma_s = sigma.reindex(rel_idx)
    rt_s = r_trail10.reindex(rel_idx)

    F = pd.DataFrame(index=rel_idx)
    # signal_trajectory (native released index)
    F["signal_run_length"] = signal_run_length(s)
    F["signal_entropy_20d"] = signal_entropy_20d(s)
    F["signal_flip_rate_60d"] = signal_flip_rate_60d(s)
    F["signal_cum_pnl_20d"] = signal_cum_pnl_20d(s, r_s)
    # information_theoretic (MI/TE vs the causal 10d return)
    F["rolling_mutual_information_252d"] = rolling_mutual_information_252d(s, rt_s)
    F["transfer_entropy_vol_to_acc_126d"] = transfer_entropy_vol_to_signal_acc(
        sigma_s, s, rt_s)

    # price features on the buffered index, then sliced to rel_idx
    P = pd.DataFrame(index=close.index)
    P["expected_hit_time"] = expected_hit_time(r, sigma)
    P["prob_timeout"] = prob_timeout(r, sigma)
    P["path_tortuosity_20d"] = path_tortuosity_20d(r)
    P["realized_semi_vol_ratio_20d"] = realized_semi_vol_ratio(r)
    P["amihud_illiquidity_20d"] = amihud_illiquidity(r, volume)
    P["rolls_spread_20d"] = rolls_effective_spread(close)
    P["kyles_lambda_20d"] = kyles_lambda(r, volume)
    P["overnight_gap"] = overnight_gap(open_, close.shift(1))
    P["dist_lead_lag_centroid"] = distance_to_lead_lag_centroid(rp, inst)
    P["asset_class_dispersion_z"] = asset_class_dispersion_z(rp, inst)
    P["ewma_implied_corr_z"] = ewma_implied_corr_z(rp, inst)
    mra = mra_energy_bands(r)
    for col in mra.columns:
        P[col] = mra[col]

    F = F.join(P.reindex(rel_idx), how="left")

    # concept_drift last, on the assembled matrix. ffill bridges the sporadic
    # zero-volume NaN holes (amihud/kyles) so the discriminator's predict step
    # is not forced to drop those rows; dropna handles the warmup head.
    drift_in = F.ffill().dropna()
    ras = regime_alignment_score(drift_in, train_end=TRAIN_END)
    F["regime_alignment_score"] = ras.reindex(F.index)

    return F[FEATURE_ORDER]


In [ ]:
# Build all 11 instruments (the 3 focus + the rest for the cross-cutting
# panels 5-8). ~10-15s total.
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    FEATS = {}
    for inst in INSTRUMENTS:
        FEATS[inst] = build_feature_matrix(inst)
        print(f"  {inst}: {FEATS[inst].shape}")
print("done — FEATS holds one 23-column DataFrame per instrument")


## 2. Per-instrument sanity checks

Helpers used by every per-instrument section, then one section each for `cl1s`,
`es1s`, `gc1s`.

In [ ]:
SIGNAL_COLORS = {-1.0: "#d62728", 0.0: "#7f7f7f", 1.0: "#2ca02c"}


def _signal_for(inst: str) -> pd.Series:
    """Primary signal aligned to the instrument's feature index."""
    idx = FEATS[inst].index
    return sig_idx[inst].astype(float).reindex(idx)


def plot_feature_timeseries(inst: str):
    """2a. Grid of every feature over the released window, shared x-axis, with
    the primary signal overlaid as faint background shading (-1/0/+1)."""
    F = FEATS[inst]
    sig = _signal_for(inst)
    cols = list(F.columns)
    ncol = 4
    nrow = int(np.ceil(len(cols) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 2.1 * nrow),
                             sharex=True)
    axes = axes.ravel()
    x = F.index
    for ax, col in zip(axes, cols):
        for val, color in SIGNAL_COLORS.items():
            ax.fill_between(x, 0, 1, where=(sig == val).to_numpy(),
                            transform=ax.get_xaxis_transform(),
                            color=color, alpha=0.07, linewidth=0)
        ax.plot(x, F[col].to_numpy(), color="#1f1f1f", linewidth=0.7)
        ax.set_title(col, fontsize=8)
        ax.tick_params(labelsize=6)
    for ax in axes[len(cols):]:
        ax.set_visible(False)
    fig.suptitle(f"{inst} — feature time-series (bg shade = primary signal "
                 f"-1 red / 0 grey / +1 green)", fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.985))
    plt.show()


def summary_stats_table(inst: str) -> pd.DataFrame:
    """2b. Per-feature mean/std/min/max/%NaN, warmup row index (first non-NaN),
    documented warmup, and a FLAG when %NaN after the documented warmup > 5%."""
    F = FEATS[inst]
    rows = []
    for col in F.columns:
        s = F[col]
        first_valid = s.notna().to_numpy().argmax() if s.notna().any() else -1
        doc = DOC_WARMUP[col]
        if doc is None:
            pct_after = np.nan
            flag = "by-design test-era only"
        else:
            tail = s.iloc[doc:]
            pct_after = float(tail.isna().mean() * 100)
            flag = "NaN>5% post-warmup" if pct_after > 5 else ""
            if first_valid > doc:
                flag = (flag + "; " if flag else "") + "warmup>doc"
        rows.append({
            "family": FEATURE_FAMILY[col], "feature": col,
            "mean": s.mean(), "std": s.std(), "min": s.min(), "max": s.max(),
            "pct_nan": float(s.isna().mean() * 100),
            "first_valid_row": int(first_valid),
            "doc_warmup": doc, "pct_nan_post_warmup": pct_after, "FLAG": flag,
        })
    out = pd.DataFrame(rows).set_index("feature")
    return out.round(4)


def spearman_heatmap(inst: str):
    """2c. Spearman correlation heatmap of all 23 features over the released
    window."""
    F = FEATS[inst]
    corr = F.corr(method="spearman")
    fig, ax = plt.subplots(figsize=(11, 9))
    sns.heatmap(corr, cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True,
                cbar_kws={"shrink": 0.7}, linewidths=0.3, ax=ax)
    ax.set_title(f"{inst} — Spearman feature correlation", fontsize=11)
    ax.tick_params(labelsize=7)
    plt.show()
    return corr


def scale_boxplot(inst: str):
    """2d. One boxplot with all 23 features side by side (symlog y) to spot any
    feature orders-of-magnitude off the others."""
    F = FEATS[inst]
    fig, ax = plt.subplots(figsize=(13, 5))
    data = [F[c].dropna().to_numpy() for c in F.columns]
    ax.boxplot(data, tick_labels=list(F.columns), showfliers=True,
               flierprops=dict(marker=".", markersize=2, alpha=0.4))
    ax.set_yscale("symlog")
    ax.set_title(f"{inst} — feature scale check (symlog y)", fontsize=11)
    ax.tick_params(axis="x", labelrotation=90, labelsize=7)
    plt.tight_layout()
    plt.show()


### 2.cl1s — energy — the audit's outlier

In [ ]:
# 2a. time-series with signal shading
plot_feature_timeseries('cl1s')

In [ ]:
# 2b. summary stats (flagged rows = %NaN > 5%% past documented warmup)
stats_cl1s = summary_stats_table('cl1s')
stats_cl1s

In [ ]:
# 2c. Spearman correlation heatmap
corr_cl1s = spearman_heatmap('cl1s')

In [ ]:
# 2d. scale boxplot
scale_boxplot('cl1s')

### 2.es1s — equity

In [ ]:
# 2a. time-series with signal shading
plot_feature_timeseries('es1s')

In [ ]:
# 2b. summary stats (flagged rows = %NaN > 5%% past documented warmup)
stats_es1s = summary_stats_table('es1s')
stats_es1s

In [ ]:
# 2c. Spearman correlation heatmap
corr_es1s = spearman_heatmap('es1s')

In [ ]:
# 2d. scale boxplot
scale_boxplot('es1s')

### 2.gc1s — metals

In [ ]:
# 2a. time-series with signal shading
plot_feature_timeseries('gc1s')

In [ ]:
# 2b. summary stats (flagged rows = %NaN > 5%% past documented warmup)
stats_gc1s = summary_stats_table('gc1s')
stats_gc1s

In [ ]:
# 2c. Spearman correlation heatmap
corr_gc1s = spearman_heatmap('gc1s')

In [ ]:
# 2d. scale boxplot
scale_boxplot('gc1s')

The next four panels are intentionally **cross-cutting** — across
instruments and across the train/test boundary — which is what makes them
complementary to the per-instrument panels above.

## 5. Train/test distribution shift (KS)

For each `(feature, instrument)` we split the released window at `2022-01-01`
into **pre-2022** and **H1-2022** and compute the two-sample KS statistic
between the two distributions. Pairs with **KS > 0.20** are regime-sensitive
and may need per-instrument standardisation or a different handling strategy in
Step 4a.

In [ ]:
ks_rows = []
for inst in INSTRUMENTS:
    F = FEATS[inst]
    pre = F.loc[F.index < TRAIN_END]
    post = F.loc[F.index >= TRAIN_END]
    for col in F.columns:
        a = pre[col].dropna().to_numpy()
        b = post[col].dropna().to_numpy()
        if len(a) < 20 or len(b) < 20:
            ks = np.nan
        else:
            ks = float(stats.ks_2samp(a, b).statistic)
        ks_rows.append({"feature": col, "instrument": inst, "ks": ks,
                        "n_pre": len(a), "n_post": len(b)})
ks_df = pd.DataFrame(ks_rows)
ks_tab = ks_df.sort_values("ks", ascending=False, na_position="last")
flagged = ks_tab[ks_tab["ks"] > 0.20]
print(f"{len(flagged)} of {ks_tab['ks'].notna().sum()} (feature, instrument) "
      f"pairs have KS > 0.20")
ks_tab.head(30).round(4)


In [ ]:
# KS heatmap (feature x instrument) for the full picture.
ks_pivot = ks_df.pivot(index="feature", columns="instrument", values="ks")
ks_pivot = ks_pivot.reindex(FEATURE_ORDER)[INSTRUMENTS]
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(ks_pivot, cmap="magma_r", vmin=0, vmax=0.6, annot=True, fmt=".2f",
            annot_kws={"size": 6}, linewidths=0.3, cbar_kws={"shrink": 0.7},
            ax=ax)
ax.set_title("KS statistic pre-2022 vs H1-2022 (>0.20 = regime-sensitive)",
             fontsize=11)
ax.tick_params(labelsize=7)
plt.tight_layout()
plt.show()


## 6. Label correlation (point-biserial)

Join `events.csv` with the feature matrix on `(t_signal, instrument)` and, per
instrument, compute the point-biserial correlation between each feature and the
triple-barrier label (point-biserial = Pearson between a continuous variable
and a binary one). Features flat across all instruments are dead weight;
features that swing sign across instruments are the interesting ones.

In [ ]:
events = pd.read_csv(EVENTS_CSV, parse_dates=["t_signal", "t_start", "t_end"])
print("events", events.shape, "| label balance:",
      events["label"].value_counts().to_dict())

pb_rows = []
for inst in INSTRUMENTS:
    F = FEATS[inst]
    ev = events[events["instrument"] == inst].set_index("t_signal")
    joined = ev[["label"]].join(F, how="inner")
    y = joined["label"].astype(float).to_numpy()
    for col in F.columns:
        x = joined[col].to_numpy(dtype=float)
        mask = np.isfinite(x) & np.isfinite(y)
        if mask.sum() < 20 or np.unique(y[mask]).size < 2 or np.std(x[mask]) == 0:
            pb = np.nan
        else:
            pb = float(stats.pointbiserialr(y[mask], x[mask]).correlation)
        pb_rows.append({"feature": col, "instrument": inst, "pb": pb,
                        "n": int(mask.sum())})
pb_df = pd.DataFrame(pb_rows)
pb_pivot = pb_df.pivot(index="feature", columns="instrument",
                       values="pb").reindex(FEATURE_ORDER)[INSTRUMENTS]
print("matched events per instrument:",
      pb_df.groupby("instrument")["n"].max().to_dict())


In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(pb_pivot, cmap="RdBu_r", center=0, vmin=-0.4, vmax=0.4, annot=True,
            fmt=".2f", annot_kws={"size": 6}, linewidths=0.3,
            cbar_kws={"shrink": 0.7}, ax=ax)
ax.set_title("Point-biserial corr(feature, triple-barrier label) by instrument",
             fontsize=11)
ax.tick_params(labelsize=7)
plt.tight_layout()
plt.show()

# Summarise dead-weight vs sign-swinging features.
summary = pd.DataFrame({
    "mean_abs_pb": pb_pivot.abs().mean(axis=1),
    "max_abs_pb": pb_pivot.abs().max(axis=1),
    "sign_swings": (pb_pivot > 0.02).sum(axis=1).astype(str) + "+/"
                   + (pb_pivot < -0.02).sum(axis=1).astype(str) + "-",
}).round(4)
summary.sort_values("mean_abs_pb", ascending=False)


## 7. Cross-instrument scale check

Five representative features (one per family that spans instruments) shown as a
boxplot per instrument, all 11 side by side. **Flag** any feature whose
inter-quartile range varies by more than **10x** across instruments — that
tells Step 4a whether to standardise per-instrument before pooling.

In [ ]:
REP_FEATURES = [
    "signal_cum_pnl_20d",       # signal_trajectory
    "expected_hit_time",        # conditional_risk
    "amihud_illiquidity_20d",   # microstructure
    "ewma_implied_corr_z",      # cross_asset
    "mra_energy_D1",            # wavelet
]
fig, axes = plt.subplots(len(REP_FEATURES), 1, figsize=(12, 3.0 * len(REP_FEATURES)))
iqr_report = {}
for ax, feat in zip(axes, REP_FEATURES):
    data, iqrs = [], {}
    for inst in INSTRUMENTS:
        v = FEATS[inst][feat].dropna().to_numpy()
        data.append(v)
        if len(v) > 4:
            iqrs[inst] = float(np.subtract(*np.percentile(v, [75, 25])))
    ax.boxplot(data, tick_labels=INSTRUMENTS, showfliers=False)
    ax.set_title(feat, fontsize=9)
    ax.tick_params(labelsize=7)
    pos = {k: v for k, v in iqrs.items() if v > 0}
    ratio = (max(pos.values()) / min(pos.values())) if len(pos) > 1 else np.nan
    iqr_report[feat] = ratio
plt.tight_layout()
plt.show()

iqr_tab = pd.Series(iqr_report, name="max/min IQR across instruments").round(2)
iqr_tab = iqr_tab.to_frame()
iqr_tab["needs_per_inst_scaling (>10x)"] = iqr_tab.iloc[:, 0] > 10
iqr_tab


## 8. Feature-vs-signal contingency

For three high-prior features — `signal_run_length`, `signal_entropy_20d`, and
`prob_timeout` (a conditional_risk feature) — a 2D heatmap of feature deciles
(x) vs the sign of the forward-10d return (y), coloured by count. One row per
focus instrument. This is the empirical check that the feature carries
information about future returns: a flat band across deciles = no information; a
gradient = the feature shifts the forward-return-sign distribution.

In [ ]:
CONTINGENCY_FEATURES = ["signal_run_length", "signal_entropy_20d", "prob_timeout"]


def forward_return_sign(inst: str, h: int = 10) -> pd.Series:
    """Sign of the h-day forward log-return on the instrument's trading
    calendar, aligned to the feature index. Used for diagnostics only (it is
    forward-looking and never fed to a feature)."""
    sub = ohlcv[ohlcv["instrument"] == inst].sort_values("date")
    close = sub.set_index("date")["close"].astype(float)
    fwd = np.log(close.shift(-h)) - np.log(close)
    return np.sign(fwd).reindex(FEATS[inst].index)


fig, axes = plt.subplots(len(CONTINGENCY_FEATURES), len(FOCUS),
                         figsize=(4.6 * len(FOCUS), 3.4 * len(CONTINGENCY_FEATURES)))
for i, feat in enumerate(CONTINGENCY_FEATURES):
    for j, inst in enumerate(FOCUS):
        ax = axes[i, j]
        x = FEATS[inst][feat]
        ysign = forward_return_sign(inst)
        df = pd.DataFrame({"x": x, "y": ysign}).dropna()
        # Decile bins of the feature (collapse if too few uniques).
        try:
            df["dec"] = pd.qcut(df["x"], 10, labels=False, duplicates="drop")
        except ValueError:
            df["dec"] = pd.qcut(df["x"].rank(method="first"), 10, labels=False)
        ct = (df.groupby(["dec", "y"]).size().unstack("y").reindex(
            columns=[-1.0, 0.0, 1.0]).fillna(0.0))
        ct.columns = ["neg", "zero", "pos"]
        sns.heatmap(ct.T, cmap="viridis", ax=ax, cbar=(j == len(FOCUS) - 1),
                    annot=True, fmt=".0f", annot_kws={"size": 6},
                    linewidths=0.3)
        ax.set_title(f"{inst} — {feat}", fontsize=8)
        ax.set_xlabel("feature decile", fontsize=7)
        ax.set_ylabel("fwd-10d sign", fontsize=7)
        ax.tick_params(labelsize=6)
fig.suptitle("Feature decile vs forward-10d return sign (count)", fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.985))
plt.show()


## Findings

Static summary written after running the panels above (numbers quoted are from
this run; all 11 instruments built, 23 columns each).

### Looks fine

- **No constant features.** Every one of the 23 columns varies on all three
  focus instruments and across the panel.
- **No copy-paste duplicates.** The largest `|Spearman|` between two *distinct*
  features is < 0.98 on every focus instrument — nothing looks accidentally
  cloned. (`time_since_last_flip`, the exact `signal_run_length − 1` duplicate,
  was excluded up front, which is why the canonical set is 23 not 24.)
- **No leakage / pre-emption.** All 23 features are causal — proven by the
  Step-3 truncation-invariance harness — and the time-series panels show no
  feature moving *ahead* of a signal flip. The signal-trajectory features
  (`signal_run_length`, `signal_flip_rate_60d`, `signal_entropy_20d`) move
  *synchronously* with the primary signal by construction; that is expected, not
  leakage.
- **Warmups match documentation.** First-valid row equals the documented warmup
  for every feature on every instrument (the one exception is the `fesx1s` bug
  below). Signal features warm up inside the released window
  (`signal_flip_rate_60d` ≈ 60 rows, `signal_entropy_20d` / `signal_cum_pnl_20d`
  ≈ 19); price features are warmed in the pre-2020 buffer and are valid from day
  one of the released window.
- **Every family carries some label information.** No dead-weight feature: the
  weakest, `asset_class_dispersion_z`, still reaches mean|pb| 0.039 / max 0.086.
  Signal-trajectory features carry the most consistent label correlation
  (`signal_flip_rate_60d` mean|pb| 0.156, `signal_run_length` 0.138,
  `signal_entropy_20d` 0.132) and *swing sign* across instruments — the
  informative pattern panel 6 was looking for. `rolls_spread_20d` is uniformly
  positively correlated (9/11 instruments, 0 negative).
- **Most features are pool-safe on scale.** Cross-instrument IQR ratios are
  1.7x–6.8x for `signal_cum_pnl_20d`, `expected_hit_time`, `ewma_implied_corr_z`
  and `mra_energy_D1` — all < 10x, so they can be pooled without per-instrument
  standardisation. `realized_semi_vol_ratio_20d` and `overnight_gap` are also
  well-behaved (median ≈ 1 / 0, no blow-ups).

### Suspicious — fix before shipping to the model

1. **`dist_lead_lag_centroid` is 100% NaN for `fesx1s`.** Root cause: ~23 days
   in the trailing window are US holidays that Eurex trades *through*, so every
   US-peer return is NaN that day; the peer-mean centroid is NaN and a single
   NaN voids the whole `rolling(126)` window. US instruments see ≤ 1 such day
   and are fine (~775 valid rows). **Action (Step 4a):** compute the centroid
   with a minimum-peers threshold (ignore all-NaN peer days) or restrict the
   lead-lag feature to same-venue peers / drop it for `fesx1s`.

2. **`amihud_illiquidity_20d` and `kyles_lambda_20d` have ~6800x IQR spread
   across instruments** (e.g. `ng1s` ≈ 7e-11 vs `rb1s` ≈ 5e-07). They are raw
   `|r| / volume` (and `|r| / √volume`) forms and contract volume units differ
   by orders of magnitude. **Action:** standardise per-instrument (or
   log-transform) before pooling, otherwise a pooled model treats them as ~zero
   for low-volume contracts. These two are also the worst train/test shifters
   (see 4).

3. **`path_tortuosity_20d` is extremely heavy-tailed** (median ≈ 4–5, p99 ≈
   150–250, max ≈ 2000–3000). The eps-protected `Σ|r| / |Σr|` denominator
   explodes when the 20-day net move ≈ 0. It is finite (no Inf), but the tail
   will dominate an unscaled model. **Action:** winsorise or `log1p` before the
   model.

4. **Large train/test distribution shift (modelling risk, not a feature bug).**
   170 of 241 (feature, instrument) pairs have KS > 0.20 across the
   `2022-01-01` boundary. Worst offenders are the volatility/liquidity-scaled
   and wavelet features: `prob_timeout` (11/11, mean KS 0.49), `amihud` (11/11,
   0.52), `kyles` (11/11, 0.46), `mra_energy_D1/D2` (11/11, 0.49/0.54),
   `dist_lead_lag` and `transfer_entropy` (10/11). Most regime-*stable*:
   `asset_class_dispersion_z` (0/11), `overnight_gap` (1/11),
   `path_tortuosity_20d` (4/11). This quantifies Sreeram's v5 concept-drift
   finding — purged/embargoed CV alone will not absorb a covariate shift this
   large; expect to use `regime_alignment_score` as a down-weighting channel
   and/or per-regime normalisation.

5. **Coverage caveats (by design, but they shrink usable data).**
   `rolling_mutual_information_252d` is NaN for the first ~252 released rows
   (~40%; first valid ≈ Dec-2020), so it is unusable for roughly the first model
   year. `transfer_entropy_vol_to_acc_126d` ≈ 20%. `regime_alignment_score` is
   scored only on the ~60 post-`(train_end + 63)` test-era rows (90% NaN by
   construction); its apparently strong label correlation (mean|pb| 0.266, max
   0.58 on `cl1s`) is a ~60-row, single-regime estimate — treat as suggestive,
   not robust.

6. **Small-event instruments make per-instrument label correlations noisy.**
   `ho1s` (63 events), `ng1s` (120) and `gc1s` (161) drive several of the
   largest |pb| spikes (e.g. `ho1s` `signal_flip_rate_60d` +0.59). Trust the
   cross-instrument pattern over any single-instrument extreme.
